# RealDrift on BPIC2012: four tiers, three seeds, drift detection and evaluation
Generates the sudden, gradual, hard and recurrent streams for seeds 0, 1 and 2, runs PELT and Version Clustering on every stream and scores them against the ground truth.
Run the cells top to bottom. Generated streams are reused when you run again, so an interrupted run continues where it stopped.

In [ ]:
import os, sys, subprocess, json, glob, shutil
import pandas as pd
import yaml
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)

DATASET = "bpic12"
TIERS   = ["sudden", "gradual", "hard", "recurrent"]
SEEDS   = [0, 1, 2]
TUNE    = False        # True tunes detector parameters on separate streams (seeds in TUNE_SEEDS), see section 4
TUNE_SEEDS = [100]
FORCE   = False        # True regenerates streams that already exist
QUICK   = False        # True is a smoke test with a few cases per instance, not for results
QUICK_MAX_CASES = 300
STREAMS = "outputs/streams"          # eval streams go to STREAMS/eval, tuning streams to STREAMS/tune
OUT     = "outputs"                  # detection_raw.csv and detection_summary.csv

def run(cmd, tail=3000):
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    out = "\n".join(l for l in (p.stdout + p.stderr).splitlines()
                    if "it/s" not in l and "kde" not in l and "warn" not in l.lower())
    print(out[-tail:])
    if p.returncode != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")

# experiment settings for this notebook, written next to the repository defaults
exp = yaml.safe_load(open("configs/experiment.yaml"))
exp.update({"configs": ["configs/bpic2012_config.yaml"], "tiers": TIERS, "seeds": SEEDS,
            "tuning_seeds": TUNE_SEEDS, "out_dir": STREAMS})
os.makedirs(OUT, exist_ok=True)
EXP = f"{OUT}/experiment_notebook.yaml"
yaml.safe_dump(exp, open(EXP, "w"))
quick = ["--max-cases-per-instance", str(QUICK_MAX_CASES)] if QUICK else []
print("tiers", TIERS, "seeds", SEEDS, "quick" if QUICK else "full scale")

## 1. Inputs
BPIC12 needs the sublogs, the generated concept pools and the calendars. If `data/calendars` is empty it is built on the first run (a few minutes).

In [ ]:
cfg = yaml.safe_load(open("configs/bpic2012_config.yaml"))
print("n_clusters", cfg["n_clusters"], "| d_supp", cfg["calendar_params"]["d_supp"], "| SA iterations", cfg["composer"]["sa_iterations"])
for d in ["data/sublogs", "data/concept_pools", "data/calendars"]:
    print(d, sorted(os.listdir(d)))

## 2. Generate the streams
Four tiers times three seeds is 12 streams. A full-scale stream takes about 3 minutes, so expect around 40 minutes in total.

In [ ]:
run([sys.executable, "scripts/run_composition.py", "--experiment", EXP, "--phase", "eval"]
    + (["--force"] if FORCE else []) + quick)

In [ ]:
manifests = [json.load(open(f)) for f in sorted(glob.glob(f"{STREAMS}/eval/{DATASET}_*_manifest.json"))]
comp = pd.DataFrame([{"tier": m["tier"], "seed": m["seed"], "n_drifts": len(m["drift_types"]),
                      "placed": m["n_placed"], "excluded": m["n_excluded"],
                      "excluded_share": round(m["n_excluded"] / max(1, m["n_placed"] + m["n_excluded"]), 3),
                      "mean_delay_h": round(m["mean_delay_per_case_h"], 1),
                      "valid": m["validation"]["ok"] and m["ground_truth_check"]["ok"]} for m in manifests])
display(comp.sort_values(["tier", "seed"]))
assert comp["valid"].all(), "a composed stream failed validation, do not evaluate detectors on it"
assert len(comp) == len(TIERS) * len(SEEDS), "not all streams were generated"

### Ground truth of one stream per tier

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(TIERS), 1, figsize=(10, 2.4 * len(TIERS)), sharex=False)
for ax, tier in zip(axes, TIERS):
    pre = f"{STREAMS}/eval/{DATASET}_{tier}_seed{SEEDS[0]}"
    log = pd.read_csv(pre + "_drift_log.csv")
    tcol = [c for c in log.columns if "time" in c.lower()][0]
    t = pd.to_datetime(log[tcol], format="mixed", utc=True).dt.tz_localize(None)
    t.dt.to_period("W").value_counts().sort_index().plot(ax=ax, color="0.4")
    tr = pd.read_csv(pre + "_transitions.csv")
    weeks = pd.period_range(t.min(), t.max(), freq="W")
    for _, r in tr.iterrows():
        s, e = pd.Timestamp(r["start_ts"]), pd.Timestamp(r["end_ts"])
        ps, pe = pd.Period(s.tz_localize(None) if s.tzinfo else s, "W"), pd.Period(e.tz_localize(None) if e.tzinfo else e, "W")
        i0, i1 = weeks.get_loc(ps) if ps in weeks else 0, weeks.get_loc(pe) if pe in weeks else 0
        ax.axvspan(i0 - .3, i1 + .3, color="red", alpha=.35)
    ax.set_title(f"{tier} (seed {SEEDS[0]}), events per week, red is ground truth", fontsize=9)
    ax.set_xlabel("")
plt.tight_layout(); plt.show()

## 3. Detectors
PELT and Version Clustering are configured in `configs/detectors.yaml`. The parameters in the `params` blocks are used as they are, unless tuning is switched on below.

In [ ]:
det = yaml.safe_load(open("configs/detectors.yaml"))
print("detectors:", list(det["detectors"]))
for k, v in det["detectors"].items():
    print(k, v["params"])
print("tolerances:", det["tolerances"])

## 4. Optional tuning
Tuning picks the grid point with the best mean F1 on separate streams (`TUNE_SEEDS`, never reported). It generates those streams first and then searches `grid` in `configs/detectors.yaml`.
Skip this and the settings in `params` are used.

In [ ]:
TUNED = f"{OUT}/tuned_params.yaml"
if TUNE:
    run([sys.executable, "scripts/run_composition.py", "--experiment", EXP, "--phase", "tune"] + quick)
    run([sys.executable, "evaluation/run_detection.py", "--phase", "tune", "--experiment", EXP, "--tuned", TUNED], tail=1500)
    print(open(TUNED).read())
elif os.path.exists(TUNED):
    os.remove(TUNED)          # make sure an old tuning result is not used silently
    print("removed an old", TUNED)

## 5. Detect and evaluate
Every detector runs on every stream. A tolerance of one window is the main setting, half a window and two windows are the sensitivity analysis. A crashed run counts as a failure (F1 of 0).

In [ ]:
run([sys.executable, "evaluation/run_detection.py", "--phase", "eval", "--experiment", EXP,
     "--tuned", TUNED, "--out-dir", OUT], tail=200)
raw = pd.read_csv(f"{OUT}/detection_raw.csv")
print("runs", len(raw) // raw["tolerance"].nunique(), "| errors", int((raw["status"] == "error").sum() // raw["tolerance"].nunique()))
if (raw["status"] == "error").any():
    display(raw[raw["status"] == "error"][["tier", "seed", "detector", "error"]].drop_duplicates())

## 6. Results (mean and sample standard deviation over seeds)

In [ ]:
s = pd.read_csv(f"{OUT}/detection_summary.csv")
def fmt(m, sd): return f"{m:.2f} ± {0 if pd.isna(sd) else sd:.2f}"
s["P"]  = [fmt(a, b) for a, b in zip(s.precision_mean, s.precision_std)]
s["R"]  = [fmt(a, b) for a, b in zip(s.recall_mean, s.recall_std)]
s["F1"] = [fmt(a, b) for a, b in zip(s.f1_mean, s.f1_std)]
for tol in s["tolerance"].unique():
    print("tolerance:", tol)
    display(s[s.tolerance == tol].pivot_table(index="tier", columns="detector", values="F1", aggfunc="first")
            .reindex(TIERS))

In [ ]:
main = s[s.tolerance == s["tolerance"].iloc[0]]
display(main[["tier", "detector", "n_runs", "n_errors", "n_gt", "P", "R", "F1"]]
        .assign(tier=lambda d: pd.Categorical(d.tier, TIERS, ordered=True)).sort_values(["tier", "detector"]))
ax = main.pivot(index="tier", columns="detector", values="f1_mean").reindex(TIERS).plot.bar(
    yerr=main.pivot(index="tier", columns="detector", values="f1_std").reindex(TIERS).fillna(0),
    figsize=(7, 3), ylim=(0, 1.05), capsize=3)
ax.set_ylabel("F1"); ax.set_xlabel(""); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### LaTeX table (F1 as mean ± std over seeds, main tolerance)

In [ ]:
piv = main.pivot_table(index="tier", columns="detector", values="F1", aggfunc="first").reindex(TIERS)
print(piv.to_latex(escape=False, column_format="l" + "c" * piv.shape[1]))